In [17]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def generate_mock_csi_data(num_samples=1000, num_subcarriers=56):
    # Generate timestamps
    start_time = datetime.now()
    timestamps = [start_time + timedelta(seconds=i) for i in range(num_samples)]
    
    # Generate random CSI values for subcarriers
    np.random.seed(0)  # For reproducibility
    csi_data = np.random.randn(num_samples, num_subcarriers)
    
    # Create a DataFrame
    columns = ['timestamp'] + [f'subcarrier_{i}' for i in range(num_subcarriers)]
    data = np.column_stack((timestamps, csi_data))
    df = pd.DataFrame(data, columns=columns)
    
    # Convert timestamps to string for CSV output
    df['timestamp'] = df['timestamp'].astype(str)
    
    # Save to CSV
    df.to_csv('mock_csi_data.csv', index=False)
    
    return df

# Generate the mock data
mock_data = generate_mock_csi_data()
print(mock_data.head())


                    timestamp subcarrier_0 subcarrier_1 subcarrier_2  \
0  2024-05-30 02:46:18.827790     1.764052     0.400157     0.978738   
1  2024-05-30 02:46:19.827790     0.066517     0.302472    -0.634322   
2  2024-05-30 02:46:20.827790    -0.861226     1.910065    -0.268003   
3  2024-05-30 02:46:21.827790      1.18803     0.316943     0.920859   
4  2024-05-30 02:46:22.827790    -1.540797     0.063262     0.156507   

  subcarrier_3 subcarrier_4 subcarrier_5 subcarrier_6 subcarrier_7  \
0     2.240893     1.867558    -0.977278     0.950088    -0.151357   
1    -0.362741     -0.67246    -0.359553    -0.813146    -1.726283   
2     0.802456     0.947252     -0.15501     0.614079     0.922207   
3     0.318728     0.856831    -0.651026    -1.034243     0.681595   
4     0.232181    -0.597316    -0.237922    -1.424061     -0.49332   

  subcarrier_8  ... subcarrier_46 subcarrier_47 subcarrier_48 subcarrier_49  \
0    -0.103219  ...     -1.252795       0.77749     -1.613898      

In [20]:
mock_data.shape

(1000, 57)

In [22]:
import pandas as pd
import numpy as np

def preprocess_csi_data():
    # Load the CSI data from the CSV file
    csi_data = mock_data

    # Extract the relevant columns: timestamp, subcarrier amplitude values
    timestamps = csi_data['timestamp']
    subcarriers = csi_data.iloc[:, 1:]  # Assuming subcarrier data starts from the 3rd column
    
    # Outlier Removal
    def remove_outliers(subcarrier_series, window_size=5, threshold=3):
        filtered_series = subcarrier_series.copy()
        for i in range(window_size, len(subcarrier_series)):
            window_data = subcarrier_series[i-window_size:i]
            mean = window_data.mean()
            std = window_data.std()
            if abs(subcarrier_series[i] - mean) > threshold * std:
                filtered_series[i] = subcarrier_series[i-1]
        return filtered_series

    subcarriers_filtered = subcarriers.apply(lambda x: remove_outliers(x))

    # Subcarrier Aggregation
    def aggregate_subcarriers(subcarrier_data, time_window=3, sampling_rate=1):
        num_samples_per_window = int(time_window / sampling_rate)
        aggregated_data = []

        for start in range(0, len(subcarrier_data), num_samples_per_window):
            window = subcarrier_data[start:start + num_samples_per_window]
            std_dev = window.std(axis=0)
            aggregated_value = std_dev.mean()
            aggregated_data.append(aggregated_value)

        return np.array(aggregated_data)

    aggregated_csi = aggregate_subcarriers(subcarriers_filtered)

    # Create a DataFrame for the aggregated CSI values
    aggregated_df = pd.DataFrame(aggregated_csi, columns=['aggregated_csi'])
    
    # Save the processed data to a new CSV file
    aggregated_df.to_csv('processed_csi_data.csv', index=False)

    return aggregated_df

processed_data = preprocess_csi_data()

print(processed_data.head())


   aggregated_csi
0        0.911194
1        0.769639
2        0.724491
3        0.768704
4        0.805220


In [21]:
processed_data.shape

(334, 1)

In [5]:
# Parameters
w1 = 5  # 5-second window for outlier detection
w2 = 3  # 3-second window for aggregation

In [15]:
def remove_outliers(df, window_size=5, threshold=3):
    df_cleaned = df.copy()
    rolling_mean = df_cleaned.groupby('subcarrier')['csi_amplitude'].rolling(window=window_size, min_periods=1).mean().reset_index(level=0, drop=True)
    rolling_std = df_cleaned.groupby('subcarrier')['csi_amplitude'].rolling(window=window_size, min_periods=1).std().reset_index(level=0, drop=True)
    
    is_outlier = (np.abs(df_cleaned['csi_amplitude'] - rolling_mean) > threshold * rolling_std)
    df_cleaned.loc[is_outlier, 'csi_amplitude'] = np.nan
    df_cleaned['csi_amplitude'] = df_cleaned.groupby('subcarrier')['csi_amplitude'].fillna(method='ffill')
    
    return df_cleaned

In [11]:
def aggregate_subcarriers(df, window_size=3):
    df_aggregated = df.copy()
    df_aggregated.set_index('timestamp', inplace=True)
    df_aggregated = df_aggregated.groupby('subcarrier')['csi_amplitude'].rolling(window=f'{window_size}s', min_periods=1).std().reset_index()
    aggregated = df_aggregated.groupby('timestamp')['csi_amplitude'].mean().reset_index(name='aggregated_csi')
    
    return aggregated

In [16]:
df_cleaned = remove_outliers(df, window_size=w1)

# Apply subcarrier aggregation
df_aggregated = aggregate_subcarriers(df_cleaned, window_size=w2)

# Display the processed data
print(df_aggregated.head())

ValueError: Can only compare identically-labeled Series objects